<a href="https://colab.research.google.com/github/abyanrizz/practicallinearalgebra/blob/main/Chapter_7_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# chapter 7


after the past two theory-heavy chapters, you feel like you just
finished an intense workout at the gym: exhausted but energized. This chapter should
feel like a bike ride through the hills in the countryside: effortful at times but offering
a fresh and inspiring perspective.
The applications in this chapter are loosely built off of those in Chapter 4. I did this to
have some common threads that bind the chapters on vectors and on matrices. And
because I want you to see that although the concepts and applications become more
complex as you progress in linear algebra, the foundations are still built on the same
simple principles such as linear weighted combinations and the dot product.
Multivariate Data Covariance Matrices
In Chapter 4, you learned how to compute the Pearson correlation coefficient as the
vector dot product between two data variables, divided by the product of the vector
norms. That formula was for two variables (e.g., height and weight); what if you have
multiple variables (e.g., height, weight, age, weekly exercise...)?
You could imagine writing a double for loop over all of the variables, and applying
the bivariate correlation formula to all pairs of variables. But that is cumbersome and
inelegant, and therefore antithetical to the spirit of linear algebra. The purpose of
this section is to show you how to compute covariance and correlation matrices from
multivariate datasets.
Let’s start with covariance. Covariance is simply the numerator of the correlation
equation—in other words, the dot product between two mean-centered variables.
Covariance is interpreted in the same way as correlation (positive when variables
move together, negative when variables move apart, zero when the variables have no

113

linear relationship), except that covariance retains the scale of the data, and therefore
is not bound by ±1.
Covariance also has a normalization factor of n − 1, where n is the number of data
points. That normalization prevents the covariance from growing larger as you sum
more data values together (analogous to dividing by N to transform a sum into an
average). Here is the equation for covariance:
ca, b = n − 1 −1 ∑
i = 1
n
xi − x yi − y

As in Chapter 4, if we call x to be the mean-centered variable x, then covariance is
simply x
T
y/ n − 1 .
The key insight to implementing this formula for multiple variables is that matrix
multiplication is an organized collection of dot products between rows of the left
matrix and columns of the right matrix.
So here’s what we do: create a matrix in which each column corresponds to each
variable (a variable is a data feature). Let’s call that matrix X. Now, the multiplication
XX is not sensible (and probably not even valid, because data matrices tend to be
tall, thus M > N). But if we were to transpose the first matrix, then the rows of
X
T
correspond to the columns of X. Therefore, the matrix product X

TX encodes all
of the pair-wise covariances (assuming the columns are mean centered, and when
dividing by n − 1). In other words, the (i,j)th element in the covariance matrix is the
dot product between data features i and j.
The matrix equation for a covariance matrix is elegant and compact:
C = X
TX
1
n − 1

Matrix C is symmetric. That comes from the proof in Chapter 5 that any matrix times
its transpose is square symmetric, but it also makes sense statistically: covariance and
correlation are symmetric, meaning that, for example, the correlation between height
and weight is the same as the correlation between weight and height.
What are the diagonal elements of C? Those contain the covariances of each variable
with itself, which in statistics is called the variance, and quantifies the dispersion
around the mean (variance is the squared standard deviation).

114 | Chapter 7: Matrix Applications

1 M. A. Redmond and A. Baveja, “A Data-Driven Software Tool for Enabling Cooperative Information Sharing
Among Police Departments,” European Journal of Operational Research 141 (2002): 660–678.
Why Transpose the Left Matrix?

There is nothing special about transposing the left matrix. If your data matrix is
organized as features-by-observations, then its covariance matrix is XXT
. If you are
ever uncertain about whether to transpose the left or right matrix, think about how
to apply the rules for matrix multiplication to produce a features-by-features matrix.
Covariance matrices are always features-by-features.
The example in the online code creates a covariance matrix from a publicly available
dataset on crime statistics. The dataset includes over a hundred features about social,
economic, educational, and housing information in various communities around the
US.1 The goal of the dataset is to use these features to predict levels of crime, but here
we will use it to inspect covariance and correlation matrices.
After importing and some light data processing (explained in the online code), we
have a data matrix called dataMat. The following code shows how to compute the
covariance matrix:
datamean = np.mean(dataMat,axis=0) # vector of feature means
dataMatM = dataMat - datamean # mean-center using broadcasting
covMat = dataMatM.T @ dataMatM # data matrix times its transpose
covMat /= (dataMatM.shape[0]-1) # divide by N-1
Figure 7-1 shows an image of the covariance matrix. First of all: it looks neat, doesn’t
it? I work with multivariate datasets in my “day job” as a neuroscience professor, and
ogling covariance matrices has never failed to put a smile on my face.
In this matrix, light colors indicate variables that covary positively (e.g., percentage
of divorced males versus number of people in poverty), dark colors indicate variables
that covary negatively (e.g., percentage of divorced males versus median income),
and gray colors indicate variables that are unrelated to each other.
As you learned in Chapter 4, computing a correlation from a covariance simply
involves scaling by the norms of the vectors. This can be translated into a matrix
equation, which will allow you to compute the data correlation matrix without for
loops. Exercise 7-1 and Exercise 7-2 will walk you through the procedure. As I wrote
in Chapter 4, I encourage you to work through those exercises before continuing to
the next section.

Multivariate Data Covariance Matrices | 115

Figure 7-1. A data covariance matrix
Final note: NumPy has functions to compute covariance and correlation matrices
(respectively, np.cov() and np.corrcoef()). In practice, it’s more convenient to use
those functions than to write out the code yourself. But—as always in this book—I
want you to understand the math and mechanisms that those functions implement.
Hence, for these exercises you should implement the covariance as a direct transla‐
tion of the formulas instead of calling NumPy functions.
Geometric Transformations via Matrix-Vector
Multiplication
I mentioned in Chapter 5 that one of the purposes of matrix-vector multiplication is
to apply a geometric transform to a set of coordinates. In this section, you will see this
in 2D static images and in animations. Along the way, you’ll learn about pure rotation
matrices and about creating data animations in Python.
A “pure rotation matrix” rotates a vector while preserving its length. You can think
about the hands of an analog clock: as time ticks by, the hands rotate but do not
change in length. A 2D rotation matrix can be expressed as:

116 | Chapter 7: Matrix Applications

T =
cos θ sin θ
−sin θ cos θ
A pure rotation matrix is an example of an orthogonal matrix. I will write more
about orthogonal matrices in the next chapter, but I would like to point out that the
columns of T are orthogonal (their dot product is cos(θ)sin(θ) − sin(θ)cos(θ)) and are
unit vectors (recall the trig identity that cos2
(θ) + sin2
(θ) = 1.)

To use this transformation matrix, set θ to some angle of clockwise rotation, and then
muliply matrix T by a 2 × N matrix of geometric points, where each column in that
matrix contains the (X,Y) coordinates for each of N data points. For example, setting
θ = 0 does not change the points’ locations (this is because θ = 0 means T = I);
setting θ = π/2 rotates the points by 90∘

around the origin.

As a simple example, consider a set of points aligned in a vertical line and the effect of
multiplying those coordinates by T. In Figure 7-2, I set θ = π/5.

Figure 7-2. Twirling points around the origin through a pure rotation matrix
Before continuing with this section, please inspect the online code that generates this
figure. Make sure you understand how the math I wrote above is translated into code,
and take a moment to experiment with different rotation angles. You can also try

Geometric Transformations via Matrix-Vector Multiplication | 117

2 Swap the minus signs in the sine functions.
to figure out how to make the rotation counterclockwise instead of clockwise; the
answer is in the footnote.2
Let’s make our investigations of rotations more exciting by using “impure” rotations
(that is, stretching and rotating, not only rotating) and by animating the transforma‐
tions. In particular, we will smoothly adjust the transformation matrix at each frame
of a movie.
There are several ways to create animations in Python; the method I’ll use here
involves defining a Python function that creates the figure content on each movie
frame, then calling a matplotlib routine to run that function on each iteration of the
movie.
I call this movie The Wobbly Circle. Circles are defined by a set of cos θ and sin θ
points, for a vector of θ angles that range from 0 to 2π.
I set the transformation matrix to the following:
T =
1 1 − φ
0 1
Why did I pick these specific values, and how do you interpret a transformation
matrix? In general, the diagonal elements scale the x-axis and y-axis coordinates,
while the off-diagonal elements stretch both axes. The exact values in the matrix
above were selected by playing around with the numbers until I found something that
I thought looked neat. Later on, and in the exercises, you will have the opportunity to
explore the effects of changing the transformation matrix.
Over the course of the movie, the value of φ will smoothly transition from 1 to 0 and
back to 1, following the formula φ = x
2
, −1 ≤ x ≤ 1. Note that T = I when φ = 1.
The Python code for a data animation can be separated into three parts. The first part
is to set up the figure:

In [ ]:
theta = np.linspace(0,2*np.pi,100)
points = np.vstack((np.sin(theta),np.cos(theta)))
fig,ax = plt.subplots(1,figsize=(12,6))
plth, = ax.plot(np.cos(x),np.sin(x),'ko')

In [ ]:
def aframe(ph):
# create and apply the transform matrix
T = np.array([ [1,1-ph],[0,1] ])
P = T@points
# update the dots' location
plth.set_xdata(P[0,:])
plth.set_ydata(P[1,:])
return plth

In [ ]:
phi = np.linspace(-1,1-1/40,40)**2
animation.FuncAnimation(fig, aframe, phi,
interval=100, repeat=True)